In [7]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

#import sys, os
#sys.path.insert(1, os.path.abspath(os.path.join("src", "alphaquant")))
#import alphaquant.run_pipeline as aq_pipeline

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))
from alphaPhosHelperFunctions import *
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
import plotly.figure_factory as ff
import kinase_library as kl
import analytics_core_V04 as ac
import matplotlib.pyplot as plt
from core import *
from matplotlib_venn import venn3

In [9]:
color_palette = ['#EEA69B', '#E78373', '#E1604C', '#DB452E', '#B3321E', '#8C2718', '#641C11']

# Data upload

In [10]:
import re
from pathlib import Path
import pandas as pd

RAW_DIR = Path('pride_data/analysis_data/revision/figure2')
INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]

def parse_filename(fname):
    """Return (workflow, condition, input_ng) from a Spectronaut report filename."""
    fname = fname.strip()
    workflow  = 'nanophos' if 'nanoPhos' in fname else ('uphos' if 'uPhos' in fname else 'unknown')
    if 'withEGF' in fname:
        condition = 'withegf'
    elif 'noEGF' in fname or 'woEGF' in fname:
        condition = 'noegf'
    elif 'HeLa' in fname:
        condition = 'hela'
    else:
        condition = 'unknown'
    m = re.search(r'(\d+)\s*ng', fname)
    return workflow, condition, int(m.group(1)) if m else None

buckets = {}
for path in sorted(RAW_DIR.glob('*.tsv')):
    w, c, n = parse_filename(path.name)
    if n is None:
        print(f"  ! skipping (no input found): {path.name}")
        continue
    buckets.setdefault(f"{w}_{c}", {})[n] = path

loaded = {}
for group, files in buckets.items():
    print(f"\n=== {group} ===")
    loaded[group] = {}
    for ng in sorted(files):
        df = pd.read_csv(files[ng], sep='\t')
        loaded[group][ng] = df
        print(f"  {ng:>5} ng: {len(df):>8,} rows  ({files[ng].name})")

nanophos_noEGF   = loaded.get('nanophos_noegf',   {})
uphos_hela       = loaded.get('uphos_hela',       {})
nanophos_withEGF = loaded.get('nanophos_withegf', {})


l_nanophos_noEGF   = [nanophos_noEGF[n]   for n in INPUT_NG_ORDER if n in nanophos_noEGF]
l_uphos_hela       = [uphos_hela[n]       for n in INPUT_NG_ORDER if n in uphos_hela]
l_nanophos_withEGF = [nanophos_withEGF[n] for n in INPUT_NG_ORDER if n in nanophos_withEGF]



  ! skipping (no input found): 20260518_120705_nanoPhos_dilser_withEGF_repeat_all_Report.tsv
  ! skipping (no input found): 20260518_162116_nanoPhos_optimization_salt_Report.tsv
  ! skipping (no input found): 20260622_091357_nanoPhos_dilser_withEGF_repeat_all_wo_norm_Report.tsv

=== nanophos_noegf ===
     10 ng:    5,461 rows  (20260506_132422_nanoPhos_dilser_woEGF_10ng_Report.tsv)
     20 ng:    6,756 rows  (20260506_132710_nanoPhos_dilser_woEGF_20ng_Report.tsv)
     50 ng:    8,282 rows  (20260506_140502_nanoPhos_dilser_woEGF_50ng_Report.tsv)
    100 ng:   18,260 rows  (20260506_140541_nanoPhos_dilser_noEGF_100ng_Report.tsv)
    200 ng:   18,847 rows  (20260506_140735_nanoPhos_dilser_noEGF_200ng_Report.tsv)
    500 ng:   27,833 rows  (20260506_141115_nanoPhos_dilser_noEGF_500ng_Report.tsv)
   1000 ng:   30,989 rows  (20260506_113358_nanoPhos_dilser_noEGF_1000ng_Report.tsv)

=== uphos_hela ===
     10 ng:      110 rows  (20260507_105509_uPhos_HeLa_10ng_Report.tsv)
     20 ng:      34

# Splitting data into working dictionaries

In [11]:

nanoPhos_woEGF   = nanophos_noEGF
nanoPhos_withEGF = nanophos_withEGF
uPhos_HeLa       = uphos_hela


l_nanoPhos_woEGF   = l_nanophos_noEGF
l_nanoPhos_withEGF = l_nanophos_withEGF
l_uPhos_HeLa       = l_uphos_hela

for name, d in [('nanoPhos_woEGF',   nanoPhos_woEGF),
                ('nanoPhos_withEGF', nanoPhos_withEGF),
                ('uPhos_HeLa',       uPhos_HeLa)]:
    print(f"  {name:<20} inputs: {sorted(d)} ng  ({len(d)} datasets)")


  nanoPhos_woEGF       inputs: [10, 20, 50, 100, 200, 500, 1000] ng  (7 datasets)
  nanoPhos_withEGF     inputs: [10, 20, 50, 100, 200, 500, 1000] ng  (7 datasets)
  uPhos_HeLa           inputs: [10, 20, 50, 100, 200, 500, 1000] ng  (7 datasets)


# Figure 2B

In [12]:
# Figure 2B — nanoPhos phosphosite depth vs protein input (unstimulated, woEGF).
#
# Source: standalone noEGF/woEGF dilution series (one report per input, n=3 reps).
# Counting logic (our agreed convention):
#   - Class I = per-(site, run) localization >= 0.75 (already applied by Spectronaut;
#     'Filtered' cells parse to NaN and are excluded).
#   - collapse_multiplicity=True -> count unique (gene, AA, pos) phosphosites, so a
#     residue seen on a singly- vs multiply-phosphorylated peptide (M1 vs M2) is ONE
#     site. This matches the localized-site convention (uPhos / Bekker-Jensen) that
#     Reviewer 2 asks for.
#   - condition is read from each column's sample NAME (not the filename), so any
#     report mixing EGF+/EGF- columns is handled safely.
import importlib, core
importlib.reload(core)
from core import class_I_by_condition
import numpy as np
import pandas as pd

woEGF_counts = class_I_by_condition(nanoPhos_woEGF, 'woegf', collapse_multiplicity=True)

rows = []
for ng in sorted(woEGF_counts):
    c = woEGF_counts[ng]
    rows.append({
        'input_ng': ng,
        'n_reps':   len(c),
        'median':   int(np.median(c)),
        'mean':     int(np.mean(c)),
        'std':      int(np.std(c, ddof=1)) if len(c) > 1 else 0,
        'min':      int(np.min(c)),
        'max':      int(np.max(c)),
        'CV%':      round(100 * np.std(c, ddof=1) / np.mean(c), 1) if len(c) > 1 else 0,
    })

summary = pd.DataFrame(rows).set_index('input_ng')
print(summary.to_string())


          n_reps  median   mean  std    min    max  CV%
input_ng                                               
10             3    1138   1154   39   1125   1199  3.4
20             3    2387   2367   74   2285   2429  3.1
50             3    4237   4205   58   4138   4240  1.4
100            3    6788   6752  176   6560   6908  2.6
200            3   10319  10415  183  10300  10627  1.8
500            3   14944  14899  256  14623  15130  1.7
1000           3   17268  17282  134  17155  17423  0.8


In [13]:
# Figure 2B box — built from the SAME woEGF_counts as the summary table above,
# so the plotted points and the reported numbers can never diverge.
import plotly.graph_objects as go
from core import _hex_to_rgba

BOX_COLOR   = '#8A0000'   # Class I deep red (alphaPhosHelperFunctions convention)
POINT_COLOR = '#393E46'
order = sorted(woEGF_counts)

fig = go.Figure()
for ng in order:
    ys = woEGF_counts[ng]
    x  = f"{ng} ng"
    fig.add_trace(go.Box(
        y=ys, x=[x] * len(ys), name=x,
        boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=POINT_COLOR, line=dict(width=0.5, color='black')),
        line=dict(color=BOX_COLOR, width=1.5),
        fillcolor=_hex_to_rgba(BOX_COLOR, 0.15),
        showlegend=False,
    ))

fig.update_layout(template='plotly_white', width=600, height=600,
                  xaxis_title='Protein input', yaxis_title='Class I phosphosites',
                  showlegend=False)
fig.update_xaxes(categoryorder='array', categoryarray=[f"{ng} ng" for ng in order])
fig.update_yaxes(range = [0,19100])
fig.show()
fig.write_image(r'figures/figure2/figure2b_box.pdf', width=600, height=600)


# Figure 2C

In [14]:
# Figure 2C — per-site dilution linearity (log2 intensity ~ log2 input).
# Multiplicity policy: KEEP (collapse_multiplicity=False). Linearity is a
# per-feature quantitative assessment, so a residue's M1 and M2 forms are fitted
# as separate dilution curves (counts collapse multiplicity; quant keeps it).
# enforce_cutoff=0.75 verifies localization per (site, run) from the
# PTM.SiteProbability columns rather than trusting the export.
import importlib, core
importlib.reload(core)
from core import calculate_dilution_linearity, plot_dilution_linearity

corr_df = calculate_dilution_linearity(
    nanoPhos_woEGF, min_dilutions=4,
    collapse_multiplicity=False,   # policy: linearity keeps multiplicity
    enforce_cutoff=0.75,
)

In [15]:
import plotly.graph_objects as go
import numpy as np

a = corr_df['r_squared'].dropna().values

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=a,
    nbinsx=len(a) // 20,            # Dynamic bin count
    name='R²',
    marker=dict(
        color='black',
        line=dict(color='black', width=0.3)
    ),
    opacity=0.8
))
fig.add_vline(x=np.nanmedian(a),
              line={'dash': 'dash', 'width': 3, 'color': 'darkred'})
fig.update_layout(
    width=600,
    height=600,
    template='plotly_white',
    xaxis_title='R squared',
    yaxis_title='Count',
    font=dict(size=12),
    showlegend=False,
    bargap=0.05
)
fig.show()
fig.write_image(r'figures/figure2/figure2c.pdf', height=600, width=600)

In [16]:
print(np.nanmedian(a))

0.9722880893720902


# Figure 2D

In [17]:
# Load selectivity values from Spectronaut reports
sel_10ng = pd.read_csv(r'pride_data/analysis_data/figure2/selectivity_10ng.tsv', sep = '\t')
sel_20ng = pd.read_csv(r'pride_data/analysis_data/figure2/selectivity_20ng.tsv', sep = '\t')
sel_50ng = pd.read_csv(r'pride_data/analysis_data/figure2/selectivity_50ng.tsv', sep = '\t')
sel_100ng = pd.read_csv(r'pride_data/analysis_data/figure2/selectivity_100ng.tsv', sep = '\t')
sel_200ng = pd.read_csv(r'pride_data/analysis_data/figure2/selectivity_200ng.tsv', sep = '\t')
sel_500ng = pd.read_csv(r'pride_data/analysis_data/figure2/selectivity_500ng.tsv', sep = '\t')
sel_1000ng = pd.read_csv(r'pride_data/analysis_data/figure2/selectivity_1000ng.tsv', sep = '\t')

In [18]:
list_sel = [sel_10ng, sel_20ng, sel_50ng, sel_100ng, sel_200ng, sel_500ng, sel_1000ng]

selectivity_values = []
for df in list_sel:
    selectivity_values.append(df.set_index('XLabel').apply(np.sum,axis = 1).tolist())

id_values = ['10ng', '20ng', '50ng', '100ng', '200ng', '500ng', '1000ng']
id_values1 = np.repeat(id_values, 3).tolist()
flattened = [item for sublist in selectivity_values for item in sublist]
df_selectivity = pd.DataFrame({'Selectivity':flattened, 'ID':id_values1})

In [19]:
# Figure 2D — phosphopeptide selectivity per input, with mean ± SD crossbar (n=3).
# NOTE: still reads the pre-revision selectivity_*.tsv files (loaded above) — these
# numbers are NOT from the Class I reanalysis yet; regenerate before final export.
import plotly.graph_objects as go

order = id_values   # ['10ng', ..., '1000ng'] from the cell above

fig = px.strip(df_selectivity, y='Selectivity', x='ID', category_orders={'ID': order})
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False)
fig.update_traces(marker=dict(size=18, color='#db4c2e', line=dict(width=0.5, color='black')))
fig.update_yaxes(range=[0, 105], showgrid=True, gridwidth=0.5, gridcolor='#F3F2F2', griddash='solid')

# mean ± SD crossbar per condition (n=3)
stats_d = df_selectivity.groupby('ID')['Selectivity'].agg(['mean', 'std']).reindex(order)
fig.add_trace(go.Scatter(
    x=stats_d.index, y=stats_d['mean'],
    error_y=dict(type='data', array=stats_d['std'].fillna(0), visible=True,
                 color='black', thickness=1, width=10),
    mode='markers',
    marker=dict(symbol='line-ew', size=26, color='black', line=dict(width=1, color='black')),
    showlegend=False, hovertemplate='mean=%{y:.1f}%<extra></extra>',
))
fig.show()
fig.write_image(r'figures/figure2/figure2d.pdf', width=600, height=600)

# Figure 2E

In [20]:
# Figure 2E — nanoPhos vs µPhos phosphosite-depth advantage per input.
# Counting: Class I, multiplicity collapsed, localization enforced (hardened counter).
#   - nanoPhos: woEGF columns only, via class_I_by_condition.
#   - µPhos:    single-condition HeLa file, counted directly.
# Plotted points: one per nanoPhos replicate, each divided by the µPhos MEAN count
#   -> 3 points per condition; their mean equals the ratio of means below. A black
#   mean ± SD crossbar is overlaid (preferred over a boxplot for n=3).
import importlib, core
importlib.reload(core)
from core import count_sites_per_sample_ptm_report, class_I_by_condition
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]
nano_counts = class_I_by_condition(nanoPhos_woEGF, 'woegf')   # {ng: [per-rep counts]}

summary_rows, ratios, labels = [], [], []
for ng in INPUT_NG_ORDER:
    if ng not in nano_counts or ng not in uPhos_HeLa:
        continue
    nano       = nano_counts[ng]
    uphos      = list(count_sites_per_sample_ptm_report(uPhos_HeLa[ng]).values())
    uphos_mean = np.mean(uphos)
    if uphos_mean <= 0:
        continue
    per_rep = [n / uphos_mean for n in nano]          # one ratio per nanoPhos replicate
    ratios += per_rep
    labels += [f"{ng} ng"] * len(per_rep)
    summary_rows.append({
        'input_ng':       ng,
        'nano_mean':      int(np.mean(nano)),
        'uphos_mean':     round(uphos_mean, 1),
        'ratio_of_means': round(np.mean(nano) / uphos_mean, 1),
    })

summary = pd.DataFrame(summary_rows).set_index('input_ng')
print(summary.to_string())

df_ratio = pd.DataFrame({'Ratio': ratios, 'ID': labels})
order = [f"{ng} ng" for ng in INPUT_NG_ORDER if ng in nano_counts and ng in uPhos_HeLa]
fig = px.strip(df_ratio, y='Ratio', x='ID', log_y=True, category_orders={'ID': order})
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False)
fig.update_traces(marker=dict(size=18, color='#B3321E', line=dict(width=0.5, color='black')))
fig.update_yaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.update_xaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.add_hline(y=1.0, line=dict(color='black', dash='dot', width=1.5))

# mean ± SD crossbar per condition (n=3)
stats_e = df_ratio.groupby('ID')['Ratio'].agg(['mean', 'std']).reindex(order)
fig.add_trace(go.Scatter(
    x=stats_e.index, y=stats_e['mean'],
    error_y=dict(type='data', array=stats_e['std'].fillna(0), visible=True,
                 color='black', thickness=1.5, width=10),
    mode='markers',
    marker=dict(symbol='line-ew', size=26, color='black', line=dict(width=1, color='black')),
    showlegend=False, hovertemplate='mean=%{y:.1f}×<extra></extra>',
))
fig.show()
fig.write_image(r'figures/figure2/figure2e.pdf', width=600, height=600)


          nano_mean  uphos_mean  ratio_of_means
input_ng                                       
10             1154        41.0            28.1
20             2367       189.3            12.5
50             4205      1538.0             2.7
100            6752      2711.7             2.5
200           10415      3763.0             2.8
500           14899      3277.3             4.5
1000          17282      8485.0             2.0


# Figure 2F

In [21]:
import importlib, core
importlib.reload(core)
from core import process_ptm_site_report

import re
from collections import defaultdict
import pandas as pd
import plotly.express as px

# Combined withEGF+woEGF dilution series, re-exported WITHOUT Spectronaut cross-run
# normalization (Cross-Run Normalization: False) — addresses Reviewer 1's point that
# automatic normalization is inappropriate when comparing across protein loads.
# Localization is already filtered at 0.75 in the export (per-run mask masks 0 cells).
df_all = pd.read_csv(
    'pride_data/analysis_data/revision/figure2/20260622_091357_nanoPhos_dilser_withEGF_repeat_all_wo_norm_Report.tsv',
    sep='\t'
)

result = process_ptm_site_report(df_all, cutoff=0.75)
site_data_all   = result['site_data']
loc_per_run_all = result['site_localization_per_run']

PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}
sample_cols = [c for c in site_data_all.columns if c not in PROC_META]

def parse_egf_ng(name):
    egf = '+' if 'withEGF' in name else '-'
    m = re.search(r'(\d+)ng', name)
    return egf, (int(m.group(1)) if m else None)

condition_to_samples = defaultdict(list)
for s in sample_cols:
    egf, ng = parse_egf_ng(s)
    if ng is None:
        print(f"  ! could not parse condition from: {s}")
        continue
    condition_to_samples[f"EGF{egf}{ng}ng"].append(s)

print(f"Detected {len(condition_to_samples)} conditions, "
      f"{sum(len(v) for v in condition_to_samples.values())} samples total")
for cond, samples in sorted(condition_to_samples.items(),
                             key=lambda x: (x[0][3], int(re.search(r'\d+', x[0]).group()))):
    print(f"  {cond:<14}  n={len(samples)}")

rename_map = {}
for cond, samples in condition_to_samples.items():
    for i, s in enumerate(sorted(samples)):
        rename_map[s] = f"{cond}{i+1:02d}"
site_data_renamed = site_data_all.rename(columns=rename_map)

AC_META_REQUIRED = {'UPD_seq', 'PTM_localization', 'Protein_group',
                    'Gene_group', 'PTM_Collapse_key'}
extras_to_drop = [c for c in PROC_META
                  if c not in AC_META_REQUIRED and c in site_data_renamed.columns]
site_data_for_ac = site_data_renamed.drop(columns=extras_to_drop)

dict_cond_all = {cond: [rename_map[s] for s in sorted(samples)]
                 for cond, samples in condition_to_samples.items()}


site_data_grouped = ac.set_condition(site_data_for_ac, dict_cond_all)
site_data_filt    = ac.filt_per_percentage(site_data_grouped, 0.7)
site_data_imp     = ac.imputation_normal_distribution(site_data_filt).reset_index()

# 8. PCA
pca_all = ac.run_pca(site_data_imp)
pca_df  = pca_all[0][0]
print(f"\nPCA explained variance: {pca_all[1]}")

#fig.write_image(r'figures/figure2/figure2f.pdf', height=600, width=600)


Dropped 6,577 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 428,019 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 44,673 → 44,568.
Final: 44,568 sites × 42 samples.
Detected 14 conditions, 42 samples total
  EGF+10ng        n=3
  EGF+20ng        n=3
  EGF+50ng        n=3
  EGF+100ng       n=3
  EGF+200ng       n=3
  EGF+500ng       n=3
  EGF+1000ng      n=3
  EGF-10ng        n=3
  EGF-20ng        n=3
  EGF-50ng        n=3
  EGF-100ng       n=3
  EGF-200ng       n=3
  EGF-500ng       n=3
  EGF-1000ng      n=3

PCA explained variance: {'x_title': 'PC1 (0.73)', 'y_title': 'PC2 (0.04)', 'group': 'group'}


In [22]:
INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]

blue_palette = ['#03045e', '#023e8a', '#0077b6', '#0096c7', '#00b4d8', '#48cae4', '#90e0ef']
red_palette  = ['#e40b0b', '#c30e0e', '#a21112', '#821415', '#611618', '#40191c', '#1f1c1f']

color_map = {}
for i, ng in enumerate(INPUT_NG_ORDER):
    color_map[f'EGF-{ng}ng'] = blue_palette[len(blue_palette) - 1 - i]
    color_map[f'EGF+{ng}ng'] = red_palette[i]


group_order = ([f'EGF-{ng}ng' for ng in INPUT_NG_ORDER] +
               [f'EGF+{ng}ng' for ng in INPUT_NG_ORDER])

fig = px.scatter(
    pca_df, x='x', y='y', color='group',
    color_discrete_map=color_map,
    category_orders={'group': group_order},
)
fig.update_layout(width=660, height=600, template='plotly_white', showlegend = False)
fig.update_traces(marker=dict(size=18, line=dict(width=1, color='black')))
fig.update_xaxes(title = pca_all[1]['x_title'])
fig.update_yaxes(title = pca_all[1]['y_title'])
fig.show()
fig.write_image(r'figures/figure2/figure2f.pdf', height=600, width=600)


In [23]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import kinase_library as kl

INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]
AC_META_REQUIRED = {'UPD_seq', 'PTM_localization', 'Protein_group',
                    'Gene_group', 'PTM_Collapse_key'}
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}

# kinase_sequence = FlankingRegion uppercased (15-mer, central residue = phospho)
site_data_for_kl = site_data_renamed.copy()
site_data_for_kl['kinase_sequence'] = (
    site_data_for_kl['PTM_flank'].astype(str).str.upper()
)

# Short collapse key: GENE_pSITE  (matches the original v00 format)
site_data_for_kl['PTM_Collapse_key_short'] = (
    site_data_for_kl['PTM_Collapse_key'].apply(lambda x: x.split('~')[1]).apply(lambda x: x.split('_')[0])
    + '_p'
    + site_data_for_kl['PTM_Collapse_key'].apply(lambda x: x.split('~')[1]).apply(lambda x: x.split('_')[1])
)

fin_res_st = {}
fin_res_pY = {}

for ng in INPUT_NG_ORDER:
    samples_egf_plus  = sorted([c for c in site_data_for_kl.columns
                                if c.startswith(f'EGF+{ng}ng')])
    samples_egf_minus = sorted([c for c in site_data_for_kl.columns
                                if c.startswith(f'EGF-{ng}ng')])
    if len(samples_egf_plus) < 2 or len(samples_egf_minus) < 2:
        print(f"Skipping {ng} ng (n+={len(samples_egf_plus)}, n-={len(samples_egf_minus)})")
        continue

    sample_cols = samples_egf_plus + samples_egf_minus
    sub = site_data_for_kl[sample_cols + list(AC_META_REQUIRED) +
                            ['kinase_sequence', 'PTM_Collapse_key_short']].copy()

    # ac.set_condition / filt / imputation — strip kinase_sequence + short_key first
    sub_for_ac = sub.drop(columns=['kinase_sequence', 'PTM_Collapse_key_short'])
    dict_cond_sub = {f'EGFplus_{ng}': samples_egf_plus,
                     f'EGFminus_{ng}': samples_egf_minus}
    sub_grouped = ac.set_condition(sub_for_ac, dict_cond_sub)
    sub_filt    = ac.filt_per_percentage(sub_grouped, 0.7)
    sub_imp     = ac.imputation_normal_distribution(sub_filt).reset_index()

    # t-test
    ttest = ac.run_ttest(sub_imp, f'EGFplus_{ng}', f'EGFminus_{ng}')
    ttest['identifier'] = (
        ttest['identifier'].apply(lambda x: x.split('~')[1]).apply(lambda x: x.split('_')[0])
        + '_p'
        + ttest['identifier'].apply(lambda x: x.split('~')[1]).apply(lambda x: x.split('_')[1])
    )
    ttest.columns = ['PTM_Collapse_key', 'T-statistics', 'pvalue', 'mean_group1', 'mean_group2',
                     'std(group1)', 'std(group2)', 'log2FC', 'test', 'correction', 'padj',
                     'rejected', 'group1', 'group2', 'FC', '-log10 pvalue', 'Method']

    # Merge ttest with kinase_sequence, format for kinase_library
    kinases = sub[['PTM_Collapse_key_short', 'kinase_sequence']].rename(
        columns={'PTM_Collapse_key_short': 'PTM_Collapse_key'}
    ).merge(
        ttest[['PTM_Collapse_key', 'T-statistics', 'log2FC', 'padj']],
        on='PTM_Collapse_key', how='inner'
    )
    kinases = kinases.rename(columns={
        'PTM_Collapse_key': 'Phosphosites',
        'kinase_sequence':  'Sequence',
        'log2FC':           'logFC',
        'T-statistics':     't',
        'padj':             'adj.P.Val',
    })[['Phosphosites', 'Sequence', 'logFC', 't', 'adj.P.Val']]

    diff_data = kl.DiffPhosData(kinases, lfc_col='logFC', seq_col='Sequence',
                                 pval_col='adj.P.Val', pval_thresh=0.1)
    fin_res_st[f'{ng}ng'] = diff_data.kinase_enrichment(
        kin_type='ser_thr', kl_method='percentile_rank', kl_thresh=15
    )
    fin_res_pY[f'{ng}ng'] = diff_data.kinase_enrichment(
        kin_type='tyrosine', kl_method='percentile_rank', kl_thresh=15
    )
    print(f"  Done {ng} ng: n_sites={len(kinases):,}")

# -------------------------------------------------------------------
# Aggregate kinases of interest across dilutions
# -------------------------------------------------------------------
list_kinases = ['ERK1', 'ERK2', 'AKT1', 'MAPKAPK2', 'MAPKAPK3', 'MAPKAPK5',
                'JNK1', 'JNK2', 'P70S6K', 'PDGFRA',
                'EGFR', 'SLK', 'HIPK1', 'HIPK2', 'HGK', 'MST1']

def collect(fin_res):
    rows = []
    for ng_key, res in fin_res.items():
        df = res.combined_enrichment_results
        df = df[df.index.isin(list_kinases)].copy()
        df['ID']     = ng_key
        df['kinase'] = df.index
        rows.append(df[['most_sig_fisher_adj_pval', 'most_sig_log2_freq_factor', 'ID', 'kinase']])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

a = pd.concat([collect(fin_res_st), collect(fin_res_pY)], ignore_index=True)

a['bubble_size'] = np.where(a['most_sig_fisher_adj_pval'] <= 0.0001, 20,
                    np.where(a['most_sig_fisher_adj_pval'] <= 0.001, 15,
                    np.where(a['most_sig_fisher_adj_pval'] <= 0.01, 10,
                    np.where(a['most_sig_fisher_adj_pval'] <= 0.1, 5, 20))))

a_filtered = a[a['most_sig_fisher_adj_pval'] <= 0.1].copy()





Calculating percentiles for upregulated sites (25 substrates)
Scoring 24 ser_thr substrates
Calculating percentile for 24 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 417.53it/s] 
                                                  

Calculating percentiles for downregulated sites (41 substrates)
Scoring 41 ser_thr substrates
Calculating percentile for 41 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 390.95it/s] 
                                                  

Calculating percentiles for background (unregulated) sites (1247 substrates)
Scoring 1213 ser_thr substrates
Calculating percentile for 1213 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 329.90it/s] 
                                                  

Calculating percentiles for upregulated sites (25 substrates)
Scoring 1 tyrosine substrates
Calculating percentile for 1 tyrosine substrates
100%|██████████| 78/78 [00:00<00:00, 847.97it/s]
                                                

Cal

d:\Projects\nanoPhos_env\src\analytics_core_V04.py:308: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy




Calculating percentiles for upregulated sites (75 substrates)
Scoring 71 ser_thr substrates
Calculating percentile for 71 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 412.53it/s] 
                                                  

Calculating percentiles for downregulated sites (23 substrates)
Scoring 22 ser_thr substrates
Calculating percentile for 22 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 418.53it/s] 
                                                  

Calculating percentiles for background (unregulated) sites (1748 substrates)
Scoring 1701 ser_thr substrates
Calculating percentile for 1701 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 379.55it/s] 
                                                  

Calculating percentiles for upregulated sites (75 substrates)
Scoring 4 tyrosine substrates
Calculating percentile for 4 tyrosine substrates
100%|██████████| 78/78 [00:00<00:00, 4691.88it/s]
                                                 

C

d:\Projects\nanoPhos_env\src\analytics_core_V04.py:308: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy




Calculating percentiles for upregulated sites (185 substrates)
Scoring 177 ser_thr substrates
Calculating percentile for 177 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 371.21it/s] 
                                                  

Calculating percentiles for downregulated sites (115 substrates)
Scoring 111 ser_thr substrates
Calculating percentile for 111 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 385.89it/s] 
                                                  

Calculating percentiles for background (unregulated) sites (4989 substrates)
Scoring 4830 ser_thr substrates
Calculating percentile for 4830 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 316.28it/s] 
                                                  

Calculating percentiles for upregulated sites (185 substrates)
Scoring 8 tyrosine substrates
Calculating percentile for 8 tyrosine substrates
100%|██████████| 78/78 [00:00<00:00, 5701.36it/s]
                                             

...


Calculating percentiles for upregulated sites (470 substrates)
Scoring 454 ser_thr substrates
Calculating percentile for 454 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 437.96it/s] 
                                                  

Calculating percentiles for downregulated sites (195 substrates)
Scoring 193 ser_thr substrates
Calculating percentile for 193 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 452.53it/s] 
                                                  

Calculating percentiles for background (unregulated) sites (12501 substrates)
Scoring 12106 ser_thr substrates
Calculating percentile for 12106 ser_thr substrates
100%|██████████| 311/311 [00:01<00:00, 278.47it/s] 
                                                  

Calculating percentiles for upregulated sites (470 substrates)
Scoring 16 tyrosine substrates
Calculating percentile for 16 tyrosine substrates
100%|██████████| 78/78 [00:00<00:00, 4673.45it/s]
                                        

d:\Projects\nanoPhos_env\src\analytics_core_V04.py:308: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy




Calculating percentiles for upregulated sites (364 substrates)
Scoring 337 ser_thr substrates
Calculating percentile for 337 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 451.45it/s] 
                                                  

Calculating percentiles for downregulated sites (180 substrates)
Scoring 167 ser_thr substrates
Calculating percentile for 167 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 448.56it/s] 
                                                  

Calculating percentiles for background (unregulated) sites (16576 substrates)
Scoring 16117 ser_thr substrates
Calculating percentile for 16117 ser_thr substrates
100%|██████████| 311/311 [00:01<00:00, 252.44it/s] 
                                                  

Calculating percentiles for upregulated sites (364 substrates)
Scoring 27 tyrosine substrates
Calculating percentile for 27 tyrosine substrates
100%|██████████| 78/78 [00:00<00:00, 4227.75it/s]
                                        

d:\Projects\nanoPhos_env\src\analytics_core_V04.py:308: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy




Calculating percentiles for upregulated sites (1831 substrates)
Scoring 1757 ser_thr substrates
Calculating percentile for 1757 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 424.00it/s] 
                                                  

Calculating percentiles for downregulated sites (719 substrates)
Scoring 708 ser_thr substrates
Calculating percentile for 708 ser_thr substrates
100%|██████████| 311/311 [00:00<00:00, 444.19it/s] 
                                                  

Calculating percentiles for background (unregulated) sites (17730 substrates)
Scoring 17278 ser_thr substrates
Calculating percentile for 17278 ser_thr substrates
100%|██████████| 311/311 [00:01<00:00, 248.71it/s] 
                                                  

Calculating percentiles for upregulated sites (1831 substrates)
Scoring 74 tyrosine substrates
Calculating percentile for 74 tyrosine substrates
100%|██████████| 78/78 [00:00<00:00, 4821.89it/s]
                                    

In [24]:
# Bubble-size bins (same thresholds as before — exposed as constants so the
# legend always matches the data)
SIZE_BINS = [
    (0.0001, 30, '<0.0001'),
    (0.001,  15, '0.001'),
    (0.01,   10, '0.01'),
    (0.1,     7, '0.1'),
]

def _bubble_size_for(p):
    for thr, sz, _ in SIZE_BINS:
        if p <= thr:
            return sz
    return np.nan      # not significant → not plotted

a_filtered = a[a['most_sig_fisher_adj_pval'] <= 0.1].copy()
a_filtered['bubble_size'] = a_filtered['most_sig_fisher_adj_pval'].apply(_bubble_size_for)

fig = go.Figure()

# --- main data trace (no legend entry) ---
fig.add_trace(go.Scatter(
    y=a_filtered['ID'],
    x=a_filtered['kinase'],
    mode='markers',
    marker=dict(
        size=a_filtered['bubble_size'],
        color=a_filtered['most_sig_log2_freq_factor'],
        colorscale='RdBu_r',
        cmin=-2, cmax=2,
        showscale=True,
        colorbar=dict(title='log2(FF)', x=1.02, len=0.45, y=0.78, yanchor='top'),
        line=dict(width=0.5, color='black'),
    ),
    hovertemplate='<b>%{y}</b><br>kinase: %{x}<br>'
                  'log2(FC): %{marker.color:.2f}<br>'
                  'Adj. p-value: %{text:.2e}<extra></extra>',
    text=a_filtered['most_sig_fisher_adj_pval'],
    showlegend=False,
))

# --- four dummy traces purely for the size legend (data points off-canvas) ---
for thr, sz, label in SIZE_BINS:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=sz, color='lightgray',
                    line=dict(width=0.5, color='black')),
        name=label,
        legendgroup='pval',
        showlegend=True,
    ))

fig.update_layout(
    xaxis_title='Kinase',
    yaxis_title='Protein input, ng',
    width=1200, height=800,
    plot_bgcolor='white',
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5,
                categoryorder='array',
                categoryarray=[f'{ng}ng' for ng in INPUT_NG_ORDER]),
    xaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5,
                categoryorder='array',
                categoryarray=list_kinases),
    legend=dict(
        title=dict(text='P-adjusted'),
        x=1.02, y=0.30, yanchor='top',
        bgcolor='rgba(255,255,255,0.9)',
        itemsizing='trace',     # use the marker size we set, not auto-scaled
    ),
)
fig.show()
fig.write_image(r'figures/figure2/figure2g.pdf', height=800, width=1200)


In [25]:
# [DISABLED for public repo — PRIDE MetaInfo export; uncomment to regenerate MetaInfo v02]
# # === PRIDE MetaInfo export — dump each Figure 2 panel's exact plotted source data ===
# # Run AFTER all panels above have executed (variables must be in scope).
# import sys; sys.path.insert(0, r"src")
# from metainfo_export import dump_panel
# from core import count_sites_per_sample_ptm_report
# import pandas as pd

# # 2b — per-raw Class I depth (woEGF dilution)
# _r = []
# for ng in sorted(nanoPhos_woEGF):
#     for rep, (samp, c) in enumerate(count_sites_per_sample_ptm_report(nanoPhos_woEGF[ng]).items(), 1):
#         _r.append({'Raw file': samp, 'Condition': f'{ng}ng', 'Replicate': rep,
#                    'Number of class I sites': int(c)})
# dump_panel(pd.DataFrame(_r), "Figure 2b")

# # 2c — per-site inter-dilution R^2
# dump_panel(corr_df, "Figure 2c")
# # 2d — phosphopeptide enrichment selectivity per replicate
# dump_panel(df_selectivity, "Figure 2d")
# # 2e — nanoPhos/uPhos Class I ratio per replicate
# dump_panel(df_ratio, "Figure 2e")
# # 2f — EGF dilution-series PCA coordinates
# dump_panel(pca_df.rename(columns={'x': 'PC1', 'y': 'PC2'}), "Figure 2f")
# # 2g — kinase enrichment bubble data
# dump_panel(a, "Figure 2g")
# print("Figure 2 panels exported to MetaInfo v02.")


  [MetaInfo] wrote 'Figure 2b'  (21 rows x 4 cols)
  [MetaInfo] wrote 'Figure 2c'  (8186 rows x 7 cols)
  [MetaInfo] wrote 'Figure 2d'  (21 rows x 2 cols)
  [MetaInfo] wrote 'Figure 2e'  (21 rows x 2 cols)
  [MetaInfo] wrote 'Figure 2f'  (42 rows x 4 cols)
  [MetaInfo] wrote 'Figure 2g'  (112 rows x 5 cols)
Figure 2 panels exported to MetaInfo v02.
